In [ ]:
!pip install transformers

In [ ]:
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
import torch

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased') # Get HF's tokenizer, so I don't have to write my own function
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2) # Get HF's pre-trained transformer, num_labels means it will be used for bin. classification

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
tokens = tokenizer("The movie was great!", return_tensors="pt") # Test out the tokenizer, pt means return a pytorch tensor instead of a list
print(tokens)

{'input_ids': tensor([[ 101, 1996, 3185, 2001, 2307,  999,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])}


In [ ]:
tokenizer.convert_ids_to_tokens([101, 1996, 3185, 2001, 2307, 999, 102]) # Get the matching tokens for the indices

['[CLS]', 'the', 'movie', 'was', 'great', '!', '[SEP]']

In [ ]:
tokens = tokenizer("I didn't enjoy the unbelievable acting")
print(f'Token IDs: {tokens['input_ids']}')
print(f'What the tokenizer "sees": {tokenizer.convert_ids_to_tokens(tokens['input_ids'])}')

Token IDs: [101, 1045, 2134, 1005, 1056, 5959, 1996, 23653, 3772, 102]
What the tokenizer "sees": ['[CLS]', 'i', 'didn', "'", 't', 'enjoy', 'the', 'unbelievable', 'acting', '[SEP]']


In [ ]:
tokens = tokenizer("The movie was fantabulous") # Let's try a word the tokenizer hasn't seen
print(tokenizer.convert_ids_to_tokens(tokens['input_ids']))
# The tokenizer splits the unknown word into various tokens like fan, ##ta, ##bu, ##lous

['[CLS]', 'the', 'movie', 'was', 'fan', '##ta', '##bu', '##lous', '[SEP]']


In [ ]:
tokens = tokenizer("Gyumri is in Armenia")
print(f'Token IDs: {tokens['input_ids']}')
print(f'What the tokenizer "sees": {tokenizer.convert_ids_to_tokens(tokens['input_ids'])}')

Token IDs: [101, 1043, 10513, 2213, 3089, 2003, 1999, 10110, 102]
What the tokenizer "sees": ['[CLS]', 'g', '##yu', '##m', '##ri', 'is', 'in', 'armenia', '[SEP]']


In [ ]:
# Load data
dataset = load_dataset('imdb')

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
# Tokenize everything
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=256) # tokenize reviews, add paddings or shorten the review if necessary

tokenized_train = dataset['train'].map(tokenize_function, batched=True) # Map the tokenize function with all the training reviews
tokenized_test = dataset['test'].map(tokenize_function, batched=True) # Same for testing reviews

# Format for PyTorch
tokenized_train.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
tokenized_test.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [ ]:
training_args = TrainingArguments( # Hyperparameters
    output_dir='./results',
    num_train_epochs=2, # Number of epochs
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy='epoch', # Evaluate the model after each epoch
    logging_steps=100,
    learning_rate=2e-5, # gamma = 0.00002
)

trainer = Trainer( # Handles training, eval, gradient computation, optimizer (AdamW), lr scheduling
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
)

trainer.train()

Epoch,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,0.235164,0.204432
2,0.152471,0.266361


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3126, training_loss=0.2138142973203653, metrics={'train_runtime': 3340.882, 'train_samples_per_second': 14.966, 'train_steps_per_second': 0.936, 'total_flos': 6577776384000000.0, 'train_loss': 0.2138142973203653, 'epoch': 2.0})

In [ ]:
results = trainer.evaluate(tokenized_test)
print(results)

{'eval_loss': 0.2663614749908447, 'eval_runtime': 390.3984, 'eval_samples_per_second': 64.037, 'eval_steps_per_second': 4.004, 'epoch': 2.0}


In [ ]:
predictions = trainer.predict(tokenized_test)
preds = predictions.predictions.argmax(axis=1)
labels = predictions.label_ids
accuracy = (preds == labels).mean()
print(f"Test accuracy: {accuracy:.4f}")

Test accuracy: 0.9236
